In [ ]:
import numpy as np
from sksurv.util import Surv
from matplotlib import pyplot as plt

# Loading Data

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

train_surv_Y = Surv.from_arrays(train_Y[:,0], train_Y[:,1])
test_surv_Y = Surv.from_arrays(test_Y[:,0], test_Y[:,1])

# CoxPHSurvivalAnalysis

In [ ]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import integrated_brier_score
import joblib

estimator_CoxPH = CoxPHSurvivalAnalysis(alpha=0.001).fit(train_X, train_surv_Y)

joblib.dump(estimator_CoxPH, "model_coxPH.joblib")

surv_funcs = estimator_CoxPH.predict_survival_function(test_X[50:60])

for fn in surv_funcs:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Relapse-Free Probability Over Time")
plt.xlabel("Months")
plt.ylabel("Probability of No Relapse")
plt.grid()

plt.show()

est_score_train = estimator_CoxPH.score(train_X, train_surv_Y)
est_score_test = estimator_CoxPH.score(test_X, test_surv_Y)

survs = estimator_CoxPH.predict_survival_function(test_X)
times = np.arange(1, 60)
preds = np.asarray([[fn(t) for t in times] for fn in survs])

brier = integrated_brier_score(train_surv_Y, test_surv_Y, preds, times)

print(brier)
print(est_score_train)
print(est_score_test)


# CoxnetSurvivalAnalysis

In [ ]:
from sksurv.linear_model import CoxnetSurvivalAnalysis

estimator_Coxnet = CoxnetSurvivalAnalysis(l1_ratio=0.99, fit_baseline_model=True).fit(train_X, train_surv_Y)

est_score_train = estimator_Coxnet.score(train_X, train_surv_Y)
est_score_test = estimator_Coxnet.score(test_X, test_surv_Y)

print(est_score_train)
print(est_score_test)

surv_funcs = estimator_Coxnet.predict_survival_function(test_X[:10])

for fn in surv_funcs:
    plt.step(fn.x, fn(fn.x), where="post")

plt.show()

## Random Survival Forest

In [ ]:
from sksurv.ensemble import RandomSurvivalForest

estimator_RSF = RandomSurvivalForest(random_state=5904).fit(train_X, train_surv_Y)
joblib.dump(estimator_RSF, "model_RSF.joblib")

est_score_train = estimator_RSF.score(train_X, train_surv_Y)
est_score_test = estimator_RSF.score(test_X, test_surv_Y)

survs = estimator_RSF.predict_survival_function(test_X)
times = np.arange(1, 60)
preds = np.asarray([[fn(t) for t in times] for fn in survs])

brier = integrated_brier_score(train_surv_Y, test_surv_Y, preds, times)

print(brier)
print(est_score_train)
print(est_score_test)

surv_funcs = estimator_RSF.predict_survival_function(test_X[50:60])

for fn in surv_funcs:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Relapse-Free Probability Over Time")
plt.xlabel("Months")
plt.ylabel("Probability of No Relapse")
plt.grid()

plt.show()